# BFT65B - Part 3: tái hiện 5 phép bằng PySpark (Colab) và so khớp với Polars

**SOURCE:** window function `partitionBy/orderBy` + `lag`, broadcast join, lazy plan và `explain()` theo bài giảng
"Big Data, Spark & ML at Scale" [2] (Problem 5-7, Solution 6b, 7a); 5 phép lấy từ `src/p3_ops.py` của repo.
**AI:** Claude viết 5 phép bằng PySpark, so khớp với kết quả Polars, lưu kế hoạch thực thi).


**Cách chạy:** Runtime > Run all (hoặc chạy từng cell từ trên xuống). Ở cell 2, chọn 3 file:
`p3_spark_input.parquet`, `p3_spark_exits.parquet` (trong `data/` của máy bạn) và `p3_polars_results.json` (trong `reports/`).
Cuối cùng notebook tải về 2 file kết quả: `p3_spark_results.json`, `p3_spark_plans.txt` (đặt vào `reports/` của repo).

In [1]:
import os
os.environ.pop("JAVA_HOME", None)      # dùng java có sẵn trên PATH
!java -version 2>&1 | grep -i "openjdk version"
!pip install -q pyspark
!pip show pyspark | grep -i "^version"

openjdk version "21.0.12.1" 2026-08-18
Version: 4.0.5


In [2]:
# Cell 2: tải 3 file từ máy lên Colab
from google.colab import files
up = files.upload()
print({k: round(len(v) / 1e6, 1) for k, v in up.items()}, "MB")

Saving p3_polars_results.json to p3_polars_results.json
Saving p3_spark_exits.parquet to p3_spark_exits.parquet
Saving p3_spark_input.parquet to p3_spark_input.parquet
{'p3_polars_results.json': 0.0, 'p3_spark_exits.parquet': 0.1, 'p3_spark_input.parquet': 50.6} MB


In [3]:
# Cell 3: khởi động Spark (chạy trên máy ảo Colab, chế độ local) và đọc dữ liệu
import os, json, time, statistics as st, contextlib, io
from pyspark.sql import SparkSession, functions as F, Window

spark = (SparkSession.builder.master("local[*]").appName("bft65b_p3")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "8")
         .getOrCreate())
print("Spark", spark.version, "| CPU của máy ảo:", os.cpu_count())

df = spark.read.parquet("p3_spark_input.parquet")
ex = spark.read.parquet("p3_spark_exits.parquet")
print("số dòng:", df.count(), "| số partition đầu vào:", df.rdd.getNumPartitions())
df.printSchema()

Spark 4.0.5 | CPU của máy ảo: 2
số dòng: 1681646 | số partition đầu vào: 2
root
 |-- ASSET: double (nullable = true)
 |-- EQ: double (nullable = true)
 |-- INTINC_Q: double (nullable = true)
 |-- INTINC_YTD: double (nullable = true)
 |-- NCLNLS: double (nullable = true)
 |-- NETINC_Q: double (nullable = true)
 |-- NETINC_YTD: double (nullable = true)
 |-- NONIX_Q: double (nullable = true)
 |-- NONIX_YTD: double (nullable = true)
 |-- ROA: double (nullable = true)
 |-- cert: long (nullable = true)
 |-- qi: long (nullable = true)



In [4]:
# Cell 4: 5 phép (cùng định nghĩa với src/p3_ops.py). Mỗi hàm chỉ MÔ TẢ kế hoạch (lazy); chưa tính gì.
QI0, QI1 = 2008 * 4, 2012 * 4 + 3
FLOWS = ["NETINC", "INTINC", "NONIX"]
OPS = ["filter_project", "yearly_ratios", "lag_features", "ytd_to_quarter", "join_exits"]
w = Window.partitionBy("cert").orderBy("qi")

def filter_project():
    q = (df.select("cert", "qi", "ASSET", "EQ", "NETINC_Q", "NCLNLS")
           .filter((F.col("qi") >= QI0) & (F.col("qi") <= QI1)))
    return q.agg(F.count(F.lit(1)).alias("n"), F.sum("ASSET").alias("s_asset"),
                 F.sum("EQ").alias("s_eq"), F.sum("NCLNLS").alias("s_nclnls"),
                 F.sum("NETINC_Q").alias("s_netinc"))

def yearly_ratios():
    t = df.select(F.floor(F.col("qi") / 4).alias("yr"),
                  (F.col("EQ") / F.col("ASSET")).alias("eq"),
                  (F.col("NCLNLS") / F.col("ASSET")).alias("npl"),
                  (F.col("NETINC_Q") / F.col("ASSET")).alias("roaq"))
    return (t.groupBy("yr")
             .agg(F.count(F.lit(1)).alias("n"), F.avg("eq").alias("eq"),
                  F.avg("npl").alias("npl"), F.avg("roaq").alias("roaq"))
             .orderBy("yr"))

def lag_features():
    q = (df.select("cert", "qi", "ASSET", "ROA")
           .withColumn("a4", F.lag("ASSET", 4).over(w))
           .withColumn("r4", F.lag("ROA", 4).over(w))
           .withColumn("q4", F.lag("qi", 4).over(w)))
    q = q.withColumn("ok", (F.col("qi") - F.col("q4")) == 4)        # đủ 4 quý liên tiếp
    q = (q.withColumn("g", F.when(F.col("ok"), F.col("ASSET") / F.col("a4") - 1))
          .withColumn("d", F.when(F.col("ok"), F.col("ROA") - F.col("r4"))))
    return q.agg(F.count("g").alias("n_valid"), F.sum("g").alias("s_growth"),
                 F.avg("g").alias("m_growth"), F.sum("d").alias("s_droa"),
                 F.avg("d").alias("m_droa"))

def ytd_to_quarter():
    q = df.select("cert", "qi", *[f"{f}_YTD" for f in FLOWS], *[f"{f}_Q" for f in FLOWS])
    q = q.withColumn("qprev", F.lag("qi", 1).over(w))
    cons = (F.col("qi") - F.col("qprev")) == 1
    aggs = []
    for f in FLOWS:
        q = q.withColumn(f"{f}_prev", F.lag(f"{f}_YTD", 1).over(w))
        q = q.withColumn(f"{f}_r",
                         F.when(F.col("qi") % 4 == 0, F.col(f"{f}_YTD"))       # Q1: giữ nguyên
                          .when(cons, F.col(f"{f}_YTD") - F.col(f"{f}_prev")))  # quý sau: trừ quý trước
        r_, s_ = F.col(f"{f}_r"), F.col(f"{f}_Q")
        mism = (r_.isNull() != s_.isNull()) | (r_.isNotNull() & s_.isNotNull() & (F.abs(r_ - s_) > 1e-6))
        aggs += [F.count(r_).alias(f"n_{f}"), F.sum(r_).alias(f"s_{f}"),
                 F.sum(mism.cast("int")).alias(f"mism_{f}")]
    return q.agg(*aggs)

def join_exits():
    j = df.select("cert", "ASSET").join(F.broadcast(ex.select("cert", "exit_type")), on="cert", how="left")
    return (j.groupBy("exit_type").agg(F.count(F.lit(1)).alias("n"), F.sum("ASSET").alias("s"))
             .orderBy("exit_type"))

QUERIES = {"filter_project": filter_project, "yearly_ratios": yearly_ratios,
           "lag_features": lag_features, "ytd_to_quarter": ytd_to_quarter, "join_exits": join_exits}
print("đã định nghĩa", list(QUERIES))

đã định nghĩa ['filter_project', 'yearly_ratios', 'lag_features', 'ytd_to_quarter', 'join_exits']


In [5]:
# Cell 5: chạy 5 phép, so khớp với Polars, đo thời gian, lưu kế hoạch thực thi
ref = json.load(open("p3_polars_results.json"))["results"]
TOL = 1e-8

def ck(op, rows):
    # chuyển kết quả (danh sách dict) thành dict số; cùng cách tính với p3_ops.ck_pl
    if op == "yearly_ratios":
        return {"n_years": len(rows), "sum_n": sum(r["n"] for r in rows), "sum_eq": sum(r["eq"] for r in rows),
                "sum_npl": sum(r["npl"] for r in rows), "sum_roaq": sum(r["roaq"] for r in rows)}
    if op == "join_exits":
        out = {}
        for r in rows:
            out[f"n_{r['exit_type']}"], out[f"s_{r['exit_type']}"] = r["n"], r["s"]
        return out
    return rows[0]

def max_rel_diff(a, b):
    return max(abs(float(a[k]) - float(b[k])) / max(1.0, abs(float(a[k])), abs(float(b[k]))) for k in a)

results, plans = {}, []
for op in OPS:
    q = QUERIES[op]()
    rows = [r.asDict() for r in q.collect()]            # chạy 1 lần để AQE chốt kế hoạch cuối
    a, b = ck(op, rows), ck(op, ref[op])
    same_keys = a.keys() == b.keys()
    diff = max_rel_diff(a, b) if same_keys else float("nan")
    ts = []
    for _ in range(4):                                   # 1 lần khởi động + 3 lần đo
        t = time.perf_counter(); QUERIES[op]().collect(); ts.append(time.perf_counter() - t)
    ts = ts[1:]
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        q.explain("formatted")
    plans.append(f"########## {op}: kế hoạch Spark (sau khi chạy, định dạng formatted)\n{buf.getvalue()}\n")
    results[op] = {"match_polars": bool(same_keys and diff < TOL), "max_rel_diff": diff,
                   "t_median_s": round(st.median(ts), 3), "t_min_s": round(min(ts), 3), "t_max_s": round(max(ts), 3)}
    print(f"{op:<16} khớp Polars: {results[op]['match_polars']} | sai lệch tương đối lớn nhất: {diff:.2e} | "
          f"Spark trung vị {results[op]['t_median_s']:.3f}s")

print("\nTất cả 5 phép khớp Polars:", all(v["match_polars"] for v in results.values()))

filter_project   khớp Polars: True | sai lệch tương đối lớn nhất: 0.00e+00 | Spark trung vị 0.897s
yearly_ratios    khớp Polars: True | sai lệch tương đối lớn nhất: 4.44e-16 | Spark trung vị 1.124s
lag_features     khớp Polars: True | sai lệch tương đối lớn nhất: 4.02e-14 | Spark trung vị 2.492s
ytd_to_quarter   khớp Polars: True | sai lệch tương đối lớn nhất: 0.00e+00 | Spark trung vị 3.422s
join_exits       khớp Polars: True | sai lệch tương đối lớn nhất: 0.00e+00 | Spark trung vị 0.887s

Tất cả 5 phép khớp Polars: True


In [6]:
# Cell 6: xem kế hoạch thực thi (đọc để giải thích trong report: Scan parquet, Filter, Exchange = shuffle,
# BroadcastHashJoin, Window, HashAggregate)
for p in plans:
    print(p)

########## filter_project: kế hoạch Spark (sau khi chạy, định dạng formatted)
== Physical Plan ==
AdaptiveSparkPlan (15)
+- == Final Plan ==
   ResultQueryStage (9)
   +- * HashAggregate (8)
      +- ShuffleQueryStage (7), Statistics(sizeInBytes=96.0 B, rowCount=2)
         +- Exchange (6)
            +- * HashAggregate (5)
               +- * Project (4)
                  +- * Filter (3)
                     +- * ColumnarToRow (2)
                        +- Scan parquet  (1)
+- == Initial Plan ==
   HashAggregate (14)
   +- Exchange (13)
      +- HashAggregate (12)
         +- Project (11)
            +- Filter (10)
               +- Scan parquet  (1)


(1) Scan parquet 
Output [5]: [ASSET#0, EQ#1, NCLNLS#4, NETINC_Q#5, qi#11L]
Batched: true
Location: InMemoryFileIndex [file:/content/p3_spark_input.parquet]
PushedFilters: [IsNotNull(qi), GreaterThanOrEqual(qi,8032), LessThanOrEqual(qi,8051)]
ReadSchema: struct<ASSET:double,EQ:double,NCLNLS:double,NETINC_Q:double,qi:bigint>

(2) Column

In [8]:
# Cell 7: lưu và tải về 2 file kết quả (đặt vào thư mục reports/ của repo)
meta = {"spark": spark.version, "cpu_cores_vm": os.cpu_count(), "master": "local[*]",
        "shuffle_partitions": 8, "input_partitions": df.rdd.getNumPartitions(), "rows": df.count()}
json.dump({"meta": meta, "results": results}, open("p3_spark_results.json", "w"), indent=1)
open("p3_spark_plans.txt", "w", encoding="utf-8").write("\n".join(plans))
from google.colab import files
files.download("p3_spark_results.json"); files.download("p3_spark_plans.txt")
print(meta)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

{'spark': '4.0.5', 'cpu_cores_vm': 2, 'master': 'local[*]', 'shuffle_partitions': 8, 'input_partitions': 2, 'rows': 1681646}
